In [ ]:
import tensorflow as tf
import larq as lq

model_name = 'wake_vision_binary_model'

# Hyperparameters
input_shape = (80, 80, 3)
batch_size = 2048
learning_rate = 0.001
epochs = 5

DATA_DIR = "WakeVision_Full"

quantized_layer_kwargs = dict(
    input_quantizer="ste_sign",
    kernel_quantizer="ste_sign",
    kernel_constraint="weight_clip",
    use_bias=False,
)

first_quantized_layer_kwargs = dict(
    kernel_quantizer="ste_sign",
    kernel_constraint="weight_clip",
    use_bias=False,
)

def build_model(input_shape):
    inputs = tf.keras.Input(shape=input_shape)

    x = tf.keras.layers.Conv2D(8, (3, 3), padding="same", use_bias=False)(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.PReLU()(x)

    x = tf.keras.layers.MaxPooling2D((2, 2))(x)
    x = lq.layers.QuantConv2D(16, (3, 3), padding="same", **first_quantized_layer_kwargs)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.PReLU()(x)

    x = tf.keras.layers.MaxPooling2D((2, 2))(x)
    x = lq.layers.QuantConv2D(24, (3, 3), padding="same", **quantized_layer_kwargs)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.PReLU()(x)

    x = tf.keras.layers.MaxPooling2D((2, 2))(x)
    x = lq.layers.QuantConv2D(32, (3, 3), padding="same", **quantized_layer_kwargs)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.PReLU()(x)

    x = tf.keras.layers.MaxPooling2D((2, 2))(x)
    x = lq.layers.QuantConv2D(40, (3, 3), padding="same", **quantized_layer_kwargs)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.PReLU()(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    x = lq.layers.QuantDense(40, **quantized_layer_kwargs)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.PReLU()(x)

    outputs = tf.keras.layers.Dense(2)(x)

    return tf.keras.Model(inputs, outputs)


model = build_model(input_shape)

case_optimizer = lq.optimizers.CaseOptimizer(
    (
        lq.optimizers.Bop.is_binary_variable,
        lq.optimizers.Bop(threshold=1e-6, gamma=1e-3),
    ),
    default_optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
)

model.compile(
    optimizer=case_optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()],
)

# ── Load datasets from local folders ──────────────────────────────────────────
# image_dataset_from_directory handles resizing, batching, and label assignment
# automatically based on subfolder names (e.g. person/ → 1, no_person/ → 0)

train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{DATA_DIR}/train",
    image_size=(input_shape[0], input_shape[1]),
    batch_size=batch_size,
    label_mode="int",       # sparse integer labels → matches SparseCategoricalCrossentropy
    shuffle=True,
    seed=42,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    f"{DATA_DIR}/val",
    image_size=(input_shape[0], input_shape[1]),
    batch_size=batch_size,
    label_mode="int",
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    f"{DATA_DIR}/test",
    image_size=(input_shape[0], input_shape[1]),
    batch_size=1,           # kept as 1 to match original
    label_mode="int",
    shuffle=False,
)

# ── Optional data augmentation (uncomment to enable) ──────────────────────────
data_augmentation = tf.keras.Sequential([
#    tf.keras.layers.RandomFlip("horizontal"),
#    tf.keras.layers.RandomRotation(0.2),
])

# Rescale pixels from [0, 255] → [0, 1] and apply augmentation
def preprocess_train(x, y):
    x = tf.cast(x, tf.float32) / 255.0
    x = data_augmentation(x, training=True)
    return x, y

def preprocess_eval(x, y):
    x = tf.cast(x, tf.float32) / 255.0
    return x, y

train_ds = train_ds.map(preprocess_train, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
val_ds   = val_ds.map(preprocess_eval,   num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
test_ds  = test_ds.map(preprocess_eval,  num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

print("Train batches:", train_ds.cardinality().numpy())

# ── Callbacks & training ───────────────────────────────────────────────────────
model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=model_name + ".tf",
    monitor='val_sparse_categorical_accuracy',
    mode='max',
    save_best_only=True,
)

model.fit(train_ds, epochs=epochs, validation_data=val_ds, callbacks=[model_checkpoint_callback])

model.save("full_precision_model.h5")
with lq.context.quantized_scope(True):
    model.save("binary_model.h5")
    weights = model.get_weights()

In [ ]:
#Post Training Quantization (PTQ)
model = tf.keras.models.load_model(model_name + ".tf")

def representative_dataset():
    for data in train_ds.rebatch(1).take(20000) :
        yield [tf.dtypes.cast(data[0], tf.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8 
converter.inference_output_type = tf.uint8
tflite_quant_model = converter.convert()

with open(model_name + ".tflite", 'wb') as f:
    f.write(tflite_quant_model)
    
#Test quantized model
interpreter = tf.lite.Interpreter(model_name + ".tflite")
interpreter.allocate_tensors()

output = interpreter.get_output_details()[0]  # Model has single output.
input = interpreter.get_input_details()[0]  # Model has single input.

correct = 0
wrong = 0

for image, label in test_ds :
    # Check if the input type is quantized, then rescale input data to uint8
    if input['dtype'] == tf.uint8:
       input_scale, input_zero_point = input["quantization"]
       image = image / input_scale + input_zero_point
       input_data = tf.dtypes.cast(image, tf.uint8)
       interpreter.set_tensor(input['index'], input_data)
       interpreter.invoke()
       if label.numpy() == interpreter.get_tensor(output['index']).argmax() :
           correct = correct + 1
       else :
           wrong = wrong + 1
print(f"\n\nTflite model test accuracy: {correct/(correct+wrong)}\n\n")